In [11]:
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd
from form990_parser import (
    parse_header,
    parse_return,
    get_org_type,
    parse_employees
)

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
xml_data_directory = Path(r'xml')

In [30]:
header_data = []
return_data = []
employee_data = []
contractor_data = []

for year_dir in xml_data_directory.iterdir():
    for period_dir in year_dir.iterdir():
        print(period_dir)
        counter = 0
        files_skipped = 0
        for xml_file in period_dir.glob('*.xml'):
            # Skip certain orgs and form types
            root = ET.parse(xml_file).getroot()
            org_type = get_org_type(root)
            if org_type not in ['501c3', '501c4', '527']:
                files_skipped += 1
                continue 

            if counter >= 50:
                break
            counter += 1

            # * Header
            header_content = parse_header(root, xml_file, org_type)
            header_data.append(header_content)

            # * Return
            if parse_return(root, 'tbd') is None:
                continue # Some files pertain to other types of returns (e.g., 990-T)
            return_content, employee_content, contractor_content = parse_return(root, 'tbd')
            return_data.append(return_content)
            
            # * Employee
            employee_data += employee_content
            
            # * Contractor
            contractor_data += contractor_content
        # print(counter) # Uncomment when you want to know the actual number of files
        print(f"# Files skipped: {files_skipped}")

xml\2020\2020_1A
# Files skipped: 0
xml\2020\2020_3A
# Files skipped: 11
xml\2024\2024_01A
# Files skipped: 69
xml\2024\2024_02A
# Files skipped: 93
xml\2025\2025_01A
# Files skipped: 113
xml\2025\2025_02A
# Files skipped: 116
xml\2025\2025_03A
# Files skipped: 193
xml\2025\2025_04A
# Files skipped: 191
xml\2025\2025_05A
# Files skipped: 294
xml\2025\2025_05B
# Files skipped: 12
xml\2025\2025_06A
# Files skipped: 142
xml\2025\2025_07A
# Files skipped: 248
xml\2025\2025_08A
# Files skipped: 135
xml\2025\2025_09A
# Files skipped: 139
xml\2025\2025_10A
# Files skipped: 162
xml\2025\2025_11A
# Files skipped: 179
xml\2025\2025_11B
# Files skipped: 12
xml\2025\2025_11C
# Files skipped: 327
xml\2025\2025_11D
# Files skipped: 136
xml\2025\2025_12A
# Files skipped: 112
xml\2026\2026_1A
# Files skipped: 112
xml\2026\2026_2A
# Files skipped: 79
xml\2026\2026_3A
# Files skipped: 178
xml\2026\2026_4A
# Files skipped: 177


In [31]:
pd.DataFrame(header_data)

,FilerEIN,TaxPeriodBeginDate,TaxPeriodEndDate,TaxYear,ReturnType,FilerBusinessNameLine1Txt,FilerBusinessNameLine2Txt,FilerPhone,FilerAddressLine1Txt,FilerCity,...,PreparerCity,PreparerStateCode,PreparerZipCode,BusinessOfficerName,BusinessOfficerTitle,BusinessOfficerPhone,BusinessOfficerSignatureDate,org_type,file_name,NumOfficers
0,742044647,2018-07-01,2019-06-30,2018,990,NATIONAL JEWISH HEALTH,NaN,3033884461,1400 JACKSON STREET,DENVER,...,NaN,NaN,NaN,Christine Forkner,Chief Financial Officer,3033981004,2020-06-02,501c3,xml\2020\2020_3A\202011559349300711_public.xml,1
1,581416325,2019-01-01,2019-12-31,2019,990,HAWTREE VOLUNTEER FIRE DEPARTMENT INC,NaN,2522573288,PO BOX 116,WISE,...,HENDERSON,NC,27536,STEVE BARNEY,CHIEF,2524325386,2020-05-27,501c3,xml\2020\2020_3A\202011559349300716_public.xml,1
2,223139262,2018-07-01,2019-06-30,2018,990,RURAL DEVELOPMENT INC,NaN,4138639781,241 MILLERS FALLS ROAD,TURNERS FALLS,...,WESTBOROUGH,MA,01581,GINA GOVONI,EXECUTIVE DIRECTOR,4138639781,2020-05-06,501c3,xml\2020\2020_3A\202011559349300726_public.xml,1
3,363244264,2018-07-01,2019-06-30,2018,990,WILL COUNTY GOVERNMENTAL LEAGUE,NaN,8152547700,15905 S FREDERICK STREET Suite 107,PLAINFIELD,...,Oakbrook Terrace,IL,601815209,JOHN NOAK,PRESIDENT,8157293535,2020-05-15,501c4,xml\2020\2020_3A\202011559349300731_public.xml,1
4,043316917,2019-01-01,2019-12-31,2019,990,I'M STILL HERE FOUNDATION INC,NaN,7815690229,10 TOWER OFFICE PARK NO 317,WOBURN,...,BOSTON,MA,02109,JACQUELINE VISCHER,TREASURER,7815690229,2020-05-21,501c3,xml\2020\2020_3A\202011559349300736_public.xml,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1145,341577595,2024-07-01,2025-06-30,2024,990,STARK STATE COLLEGE FOUNDATION,NaN,3304946170,6200 Frank Ave NW,North Canton,...,NaN,NaN,NaN,Joseph Richards,Comptroller,3304946170,2026-03-05,501c3,xml\2026\2026_4A\202600869349300520_public.xml,1
1146,721362424,2025-01-01,2025-12-31,2025,990,NORCO CIVIC ASSOCIATION,NaN,9857649696,PO BOX 22,NORCO,...,NORCO,LA,70079,CLARENCE DUPEPE,TREASURER,9857649696,2026-03-24,501c4,xml\2026\2026_4A\202600869349300525_public.xml,1
1147,980142381,2025-01-01,2025-12-31,2025,990,BETHSHEAN MEXICO MISSION INC,NaN,7062555330,PO BOX 7391,ATHENS,...,HARTWELL,GA,30643,DAN GILLAND,SEC/TREASURER,NaN,2026-03-12,501c3,xml\2026\2026_4A\202600869349300535_public.xml,1
1148,631288598,2025-07-01,2025-09-30,2025,990,C A S A OF FLORENCE LAUDERDALE,NaN,2567650041,102 JACKSON COURT,SHEFFIELD,...,FLORENCE,AL,35630,HEATHER BEGLEY,DIRECTOR,2567650041,2026-03-11,501c3,xml\2026\2026_4A\202600869349300540_public.xml,1


In [32]:
pd.DataFrame(return_data).head()

,PrincipalOfficer,PrincipalOfficerAddress,PrincipalOfficerCity,PrincipalOfficerStateCode,PrincipalOfficerZipCode,GrossReceiptsAmt,ActivityOrMission,MissionDesc,FormationYear,VotingMembersGoverningBodyCnt,...,OfficerMailingAddressInd,CompensationProcessCEOInd,CompensationProcessOtherInd,TotalReportableCompFromOrgAmt,TotReportableCompRltdOrgAmt,TotalOtherCompensationAmt,IndivRcvdGreaterThan100KCnt,FormerOfcrEmployeesListedInd,TotalCompGreaterThan150KInd,CompensationFromOtherSrcsInd
0,Christine Forkner,1400 Jackson Street,Denver,CO,80206,326863373,National Jewish Health's mission since 1899 is...,National Jewish Health's mission since 1899 is...,1978,44,...,0,1,1,8746402,0,0,344,1,1,1
1,STEVE BARNEY,PO BOX 116,WISE,NC,27594,107616,TO PROVIDE FIRE DEPARTMENT SERVICES FOR THE CO...,TO PROVIDE FIRE DEPARTMENT SERVICES FOR THE CO...,1981,9,...,0,0,0,0,0,0,0,0,0,0
2,GINA GOVONI,241 MILLERS FALLS ROAD,TURNERS FALLS,MA,01376,301633,"IT IS THE MISSION OF RURAL DEVELOPMENT, INC. (...",IT IS THE MISSION OF RDI TO ADVANCE THE RIGHT ...,1991,8,...,0,0,0,3782,0,0,0,0,0,0
3,JOHN NOAK,15905 S FREDRICK STREET 107,PLAINFIELD,IL,60586,740025,GOVERNMENT SERVICE,THE WILL COUNTY GOVERNMENTAL LEAGUE IS ORGANIZ...,1983,9,...,false,true,false,109475,0,34159,1,false,false,false
4,JACQUELINE VISCHER,10 TOWER OFFICE PARK NO 317,WOBURN,MA,01801,18632,TO BRING ABOUT COMMUNITY TRANSFORMATION FOR TH...,TO BRING ABOUT COMMUNITY TRANSFORMATION FOR TH...,1995,6,...,0,0,0,0,0,0,0,0,0,0


In [33]:
pd.DataFrame(employee_data)

,EIN,PersonNm,TitleTxt,AverageHoursPerWeekRt,AverageHoursPerWeekRltdOrgRt,IndividualTrusteeOrDirectorInd,ReportableCompFromOrgAmt,ReportableCompFromRltdOrgAmt,OtherCompensationAmt,OfficerInd,KeyEmployeeInd,HighestCompensatedEmployeeInd,FormerOfcrDirectorTrusteeInd,BusinessName,InstitutionalTrusteeInd
0,tbd,Jandel Allen-Davis MD,"Member, BOD",2,0,X,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,tbd,Sue Allon,"Member, BOD",2,0,X,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,tbd,Steve Arent,"Lifetime Member, BOD",0,0,X,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,tbd,Richard Baer,"Member, BOD",2,0,X,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,tbd,Geoffrey Barker,"Member, BOD",2,0,X,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12101,tbd,MARK WOOD,PRESIDENT,NaN,NaN,NaN,0,0,0,X,NaN,NaN,NaN,NaN,NaN
12102,tbd,STEPHEN BARNARD,PRESIDENT,1.00,NaN,X,0,0,0,X,NaN,NaN,NaN,NaN,NaN
12103,tbd,ANITA LEMOS,V.P.,1.00,NaN,X,0,0,0,X,NaN,NaN,NaN,NaN,NaN
12104,tbd,BRYAN GILES,TREASURER,1.00,NaN,X,0,0,0,X,NaN,NaN,NaN,NaN,NaN


In [34]:
pd.DataFrame(contractor_data)

,EIN,BusinessNameLine1Txt,AddressLine1Txt,AddressLine2Txt,CityNm,StateAbbreviationCd,ZIPCd,ServicesDesc,CompensationAmt,PersonNm,CountryCd,BusinessNameLine2Txt,ProvinceOrStateNm,ForeignPostalCd,ContractorAddress
0,tbd,Dimassimo,220 E 23rd Street,2nd Floor,New York,NY,10010,Advertising and Professional Fees,2203123,NaN,NaN,NaN,NaN,NaN,NaN
1,tbd,Siemens Medical Solutions USA Inc,51 Valley Stream Pkwy,NaN,Malvern,PA,19355,Equipment Maintenance Contract,1023832,NaN,NaN,NaN,NaN,NaN,NaN
2,tbd,HSS,MAIN,PO BOX 17033,Denver,CO,80217,Security Support,867767,NaN,NaN,NaN,NaN,NaN,NaN
3,tbd,ARUP Laboratories,MAIN,PO Box 27964,Salt Lake City,UT,84127,Lab Services,848779,NaN,NaN,NaN,NaN,NaN,NaN
4,tbd,Mindset Direct,12110 Sunset Hills Rd 600,NaN,Reston,VA,20190,Fundraising Servicses,612348,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
391,tbd,KENNETH MATTUS,1114 MAPLE STREET,NaN,SANTA MONICA,CA,90405,MUSIC PROGRAM SUPPORT,127140,NaN,NaN,NaN,NaN,NaN,NaN
392,tbd,Tuzinsky Consulting LLC,13741 Labelle St,NaN,Oak Park,MI,48237,Consulting,176325,NaN,NaN,NaN,NaN,NaN,NaN
393,tbd,MTAN Solutions LLC,966 Torke Terrace,NaN,Plymouth,MI,48170,Consulting,107250,NaN,NaN,NaN,NaN,NaN,NaN
394,tbd,Financial One Accounting Inc,44744 Helm St,NaN,Plymouth,MI,48170,Accounting Services,101076,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
NAMESPACE = '{http://www.irs.gov/efile}'
RETURN_HEADER_PATH = f'./{NAMESPACE}ReturnHeader'
PREPARER_FIRM_GROUP_PATH = f'./{NAMESPACE}PreparerFirmGrp'
FILER_PATH = f'./{NAMESPACE}Filer'
RETURN_DATA_PATH = f'./{NAMESPACE}ReturnData'
IRS990_PATH = RETURN_DATA_PATH + f'/{NAMESPACE}IRS990'

target = f'{NAMESPACE}SubjToTaxRmnrtnExPrchtPymtInd'

for year_dir in xml_data_directory.iterdir():
    for period_dir in year_dir.iterdir():
        print(period_dir)
        counter = 0
        files_skipped = 0
        for xml_file in period_dir.glob('*.xml'):
            # Skip certain orgs and form types
            root = ET.parse(xml_file).getroot()
            org_type = get_org_type(root)
            if org_type not in ['501c3', '501c4', '527']:
                files_skipped += 1
                continue 
            
            if '202011559349301216' in str(xml_file):
                continue
            
            _990 = root.find(IRS990_PATH)
            target_text = _990.find(target).text
            if target_text not in ['false', '0', 'o', 'O']:
                print(xml_file)
                raise TypeError

xml\2020\2020_1A
xml\2020\2020_3A
xml\2020\2020_3A\202011559349300711_public.xml


TypeError: 